# TP 2 — Comprendre et implémenter l'auto-attention

**Travail réalisé par : Emine OULD AGATT**

Le parcours commence par une moyenne uniforme des représentations précédentes, puis conduit à une attention dont les coefficients sont appris par le modèle.

Un principe doit être respecté à chaque étape : les coefficients d'une ligne décrivent la contribution des positions accessibles à la position courante, et leur somme doit être égale à 1.


In [8]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 — Calculer une moyenne à l'aide des matrices

In [9]:
torch.manual_seed(42)

matrice_a = torch.tril(torch.ones(3, 3))
matrice_a = matrice_a / torch.sum(matrice_a, 1, keepdim=True)

matrice_b = torch.randint(0, 10, (3, 2)).float()
resultat = matrice_a @ matrice_b

print(matrice_a)
print(matrice_b)
print(resultat)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


**À calculer :** déterminer `c₂` sans exécuter Python. Pour quelle raison chaque ligne de `a` est-elle normalisée à 1 ?

**Éléments de réponse :**
- La deuxième ligne de `c` correspond à la moyenne des deux premières lignes de `b`. On obtient donc `(2 + 6) / 2 = 4` et `(7 + 4) / 2 = 5,5`, soit `[4.0, 5.5]`. La troisième ligne procède de façon analogue avec trois vecteurs.
- Chaque ligne est divisée par la somme de ses coefficients. On calcule ainsi une moyenne pondérée plutôt qu'une somme brute dont l'amplitude augmenterait avec le nombre de positions.


## Étape 2 — Restreindre l'agrégation aux tokens accessibles

In [10]:
torch.manual_seed(1337)
batch_size, temps, dim = 4, 8, 2
x = torch.randn(batch_size, temps, dim)

x_moyen = torch.zeros((batch_size, temps, dim))
for b in range(batch_size):
    for t in range(temps):
        historique = x[b, :t + 1]
        x_moyen[b, t] = torch.mean(historique, dim=0)

print(historique)

tensor([[ 1.6455, -0.8030],
        [ 1.3514, -0.2759],
        [-1.5108,  2.1048],
        [ 2.7630, -1.7465],
        [ 1.4516, -1.5103],
        [ 0.8212, -0.2115],
        [ 0.7789,  1.5333],
        [ 1.6097, -0.4032]])


**Questions :** quelles sont les dimensions de `xprev` pour `t = 0`, `t = 3` et `t = 7` ? Pourquoi choisir l'axe `dim=0` pour la moyenne ?

**Éléments de réponse :**
- Une tranche d'historique contient `t + 1` positions et deux caractéristiques : sa forme est donc `(t + 1, 2)`. Les cas demandés donnent `(1, 2)`, `(4, 2)` et `(8, 2)`.
- L'axe `dim=0` représente le temps. Le réduire revient à calculer une représentation synthétique unique, de dimension 2, pour tous les tokens déjà disponibles.


## Étapes 3 et 4 — Remplacer les boucles par un calcul vectorisé

Une matrice triangulaire permet d'effectuer simultanément les agrégations, sans parcourir les positions avec une boucle Python.

### Variante 2 — Multiplication matricielle

In [11]:
poids_att = torch.tril(torch.ones(temps, temps))
poids_att = poids_att / poids_att.sum(1, keepdim=True)
x_moyen2 = poids_att @ x

print("Même résultat ?", torch.allclose(x_moyen, x_moyen2, atol=1e-6))

Même résultat ? True


### Variante 3 — Masquage causal suivi de Softmax

In [12]:
masque = torch.tril(torch.ones(temps, temps))
poids_softmax = torch.zeros((temps, temps))
poids_softmax = poids_softmax.masked_fill(masque == 0, float('-inf'))
poids_softmax = F.softmax(poids_softmax, dim=-1)

x_moyen3 = poids_softmax @ x
print("Même résultat ?", torch.allclose(x_moyen, x_moyen3, atol=1e-6))
print(poids_softmax)

Même résultat ? True
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


**À comprendre :** comment le produit `wei @ x` combine-t-il une matrice `(T, T)` et un tenseur `(B, T, C)` ? Quel est l'effet de `−∞` avant l'application de Softmax ?

**Éléments de réponse :**
- `@` applique le produit matriciel sur les deux derniers axes. PyTorch diffuse les coefficients `(T, T)` sur les `B` exemples : la même matrice de pondération est utilisée pour chaque élément du batch.
- L'exponentielle de `−∞` vaut zéro. Les positions masquées reçoivent donc un coefficient nul après Softmax, ce qui interdit à la position courante de prendre en compte des tokens futurs.


## Étape 5 — Déterminer les coefficients d'attention

In [13]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
taille_tete = 16

couche_cle = nn.Linear(C, taille_tete, bias=False)
couche_requete = nn.Linear(C, taille_tete, bias=False)
couche_valeur = nn.Linear(C, taille_tete, bias=False)

cle = couche_cle(x)
requete = couche_requete(x)

scores = requete @ cle.transpose(-2, -1)
scores = scores * (taille_tete ** -0.5)
scores = scores.masked_fill(masque == 0, float('-inf'))
scores = F.softmax(scores, dim=-1)

valeur = couche_valeur(x)
sortie_attention = scores @ valeur

**À expliquer :** quelle ligne de `wei[0]` représente le dernier token ? Pourquoi agréger les valeurs `v` plutôt que les clés `k` ?

**Éléments de réponse :**
- Le dernier token correspond à la ligne d'indice `7`. Comme aucune position ne lui est future dans cette séquence, il peut attribuer un poids à chacun des huit tokens.
- Les requêtes `q` et les clés `k` définissent les affinités entre positions. Les vecteurs `v` portent, eux, le contenu à agréger selon ces affinités : ce sont donc les valeurs qui constituent la sortie de l'attention.


## Étape 6 — Mettre les scores à l'échelle avant Softmax

In [14]:
k_test = torch.randn(B, T, taille_tete)
q_test = torch.randn(B, T, taille_tete)

score_brut = q_test @ k_test.transpose(-2, -1)
score_ajuste = score_brut * (taille_tete ** -0.5)

print("Variance avant:", score_brut.var().item())
print("Variance après:", score_ajuste.var().item())

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(torch.softmax(z, dim=-1))
print(torch.softmax(z * 8, dim=-1))

Variance avant: 19.847789764404297
Variance après: 1.2404868602752686
tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


**Bilan — TP 2 :** contrôler que chaque ligne de `wei` somme à 1, que les positions futures ont un poids nul et que la sortie possède les dimensions attendues. Préciser les fonctions respectives de Q, K et V.

**Synthèse des vérifications :**
- La normalisation Softmax garantit une somme des poids égale à 1 pour chaque ligne.
- Les coefficients situés au-dessus de la diagonale sont nuls : le masque maintient ainsi la causalité.
- La forme obtenue en sortie est `(4, 8, 16)`, c'est-à-dire `(batch, temps, dimension de tête)`.
- Les trois projections jouent des rôles distincts :
  - **Q (query)** exprime l'information recherchée par une position ;
  - **K (key)** représente les caractéristiques auxquelles les requêtes sont comparées ;
  - **V (value)** fournit les informations effectivement transmises après pondération.
- Sans mise à l'échelle, la variance des scores est d'environ 16 pour cette dimension de tête. La division par `√16` la ramène vers 1, ce qui empêche Softmax de devenir trop concentré sur une seule position et favorise des gradients exploitables pendant l'apprentissage.
